## Middlewares

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

google_api_key = os.getenv("GOOGLE_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

In [3]:
import langchain
langchain.__version__

'1.3.1'

In [15]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("messages", 5),
            keep=("messages", 2)
        )
    ]
)

In [12]:
# Run with thread id
config={"configurable": {"thread_id": "test-1"}}

In [ ]:
questions = [
    "What is Python?",
    "What is a variable?",
    "What is a function?",
    "What is a list in Python?",
    "What is a dictionary?",
    "What is a loop?",
    "What is an API?",
    "What is object-oriented programming?",
    "What is a database?",
    "What is machine learning?"
] 

In [16]:
for q in questions:
    response = agent.invoke(
        {"messages": [HumanMessage(content=q)]},
        config
    )

    print("Question:", q)
    print("Answer:", response["messages"][-1].content)
    print("Total Messages:", len(response["messages"]))
    print("-" * 50)

Question: What is Python?
Answer: <think>
Okay, I need to explain what Python is. Let me start by recalling what I know. Python is a programming language, right? It's pretty popular. I think it's used for a lot of different applications. But I should be specific.

First, I should mention that Python is a high-level programming language. High-level means it's easier for humans to read and write compared to low-level languages like C or assembly. High-level languages abstract away a lot of the complexities of the computer's hardware.

Python is known for its simplicity and readability. The syntax is clean, with a focus on indentation using whitespace, which is different from other languages that use braces. That's a key point. I should explain that Python uses indentation to define code blocks, which enforces a consistent style.

It's also interpreted, I think. Wait, actually, Python is executed through an interpreter. When you run a Python script, the interpreter translates the code int

Based on  tokensize 

In [18]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

@tool
def search_hotels(city: str) -> str:
    """Return hotel names for a city. Use this tool only once."""
    
    return f"Hotels found in {city}: Grand Hotel, Royal Palace Hotel, City View Hotel, Ocean Breeze Resort, Comfort Inn Downtown"

agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
    SummarizationMiddleware(
        model="groq:llama-3.1-8b-instant",
        trigger=("messages", 5),
        keep=("messages", 1)
    )
],
)

config={"configurable": {"thread_id": "test-1"}}

def count_tokens(messages):
    total_chars= sum(len(str(m.content)) for m in messages)
    return total_chars

In [19]:
system = SystemMessage(
    content="""
    You only have access to the search_hotels tool.

    Never call:
    - brave_search
    - web_search
    - google_search
    - any other tool

    If the user provides a country instead of a city,
    ask for a city name.
    """
)

In [20]:
cities= [
    "Delhi", 
    "Mumbai",
    "Bulandshahr", "NOida", "Ghaziabad "
]

for city in cities:
    response = agent.invoke(
    {
        "messages": [
            system,
            HumanMessage(content=f"find hotels in {city}")
        ]
    },
    config=config
)
    tokens = count_tokens(response["messages"])
    print(f"{city} ~ {tokens} tokens, {len(response['messages'])} messages")
    print(f"{response['messages']}")
    # print(response["messages"][-1].content)

Delhi ~ 500 tokens, 5 messages
[SystemMessage(content='\n    You only have access to the search_hotels tool.\n\n    Never call:\n    - brave_search\n    - web_search\n    - google_search\n    - any other tool\n\n    If the user provides a country instead of a city,\n    ask for a city name.\n    ', additional_kwargs={}, response_metadata={}, id='9a4b4533-03b6-495c-b5d0-fe4815966fd3'), HumanMessage(content='find hotels in Delhi', additional_kwargs={}, response_metadata={}, id='df508237-530a-4da2-8a71-8cde21eab8fc'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'ndgn9krq0', 'function': {'arguments': '{"city":"Delhi"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 285, 'total_tokens': 301, 'completion_time': 0.423524217, 'completion_tokens_details': None, 'prompt_time': 0.018092824, 'prompt_tokens_details': None, 'queue_time': 0.054040546, 'total_time': 0.441617041}, 'model_name': 'llama-3.1